![BasQ banner](logo_cropped.png)

# <b>Ground-State Energy of a Molecule with VQE </b>
**Author:** Benjamin Tirado  
**Created for:** BasQ Qiskit Fall Fest 2026 — Basque Quantum (BasQ)

<a id="goal"></a>
<div class="alert alert-block alert-success">
    
<b>Goal of this notebook: </b> Compute the **ground-state energy of the hydrogen molecule ($H_2$)** at its equilibrium bond distance using the **Variational Quantum Eigensolver (VQE)**. We will build the full workflow step by step: describe the molecule, map its electronic structure onto qubits with the **Jordan-Wigner** transformation, prepare a **UCCSD** ansatz on top of a **Hartree-Fock** reference state, and optimize it to approximate the lowest energy. By the end, you will have a working template for the electronic-structure problem that you can extend to other molecules and geometries.
</div>

# <b>Table of Contents</b>

* [Background](#background)
* [Pre-requisites](#prereq)
* [Defining the Molecule](#molecule)
* [Mapping to Qubits: Jordan-Wigner](#mapping)
* [The Ansatz: Hartree-Fock + UCCSD](#ansatz)
* [Running VQE](#vqe)
* [Results and Evaluation](#res)
* [Moving forward](#move)
* [Useful resources](#use)

## <b>Background </b> <a id="background"></a>

One of the most natural applications of a quantum computer is simulating **quantum systems** themselves. In quantum chemistry, a central task is to find the **ground-state energy** of a molecule: the lowest possible energy of its electrons for a fixed arrangement of nuclei. This single number governs much of a molecule's chemistry, from its stability to the shape of its bonds.

Classically, the cost of solving the electronic Schrödinger equation exactly grows extremely fast with the number of electrons, quickly becoming intractable. This is where quantum computers offer a promising route: the state of a molecule can be encoded into qubits, and its energy estimated by preparing trial states and measuring them.

The **Variational Quantum Eigensolver (VQE)** is a hybrid quantum-classical algorithm built for exactly this task. It relies on the **variational principle**, which guarantees that for any trial state $|\psi(\vec{\theta})\rangle$,

$$
E(\vec{\theta}) = \frac{\langle \psi(\vec{\theta}) | \hat{H} | \psi(\vec{\theta}) \rangle}{\langle \psi(\vec{\theta}) | \psi(\vec{\theta}) \rangle} \;\geq\; E_0,
$$

where $\hat{H}$ is the molecular Hamiltonian and $E_0$ is the true ground-state energy. In other words, no trial state can ever give an energy *below* the ground state. VQE exploits this by parameterizing the trial state with a quantum circuit (the **ansatz**) and using a **classical optimizer** to tune the parameters $\vec{\theta}$ until the measured energy is as low as possible.

This idea underpins one of IBM Quantum's landmark demonstrations, *Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets* (Kandala et al., Nature, 2017), which computed ground-state energies of small molecules on real superconducting hardware. In this tutorial we take the same conceptual workflow but keep it deliberately simple, working with H$_2$ on a simulator so that each ingredient is easy to inspect.

# <b>Pre-requisites </b> <a id="prereq"></a>
For starters, make sure you have installed the Qiskit SDK and its supporting modules: `qiskit_aer` (for noiseless simulations) and `qiskit_ibm_runtime` (for real-hardware experiments), as well as the visualization package `qiskit[visualization]`. For the chemistry-specific parts we use `qiskit_nature`, which builds the molecular problem and the fermion-to-qubit mappings, together with the classical chemistry backend `pyscf` that computes the one- and two-electron integrals. We also use `qiskit_algorithms` for the VQE routine and its optimizers.

> **Note:** `qiskit_nature` needs a classical electronic-structure driver to obtain the molecular integrals. Here we use PySCF via `PySCFDriver`. On some systems (e.g. native Windows) PySCF can be tricky to install; if so, running inside WSL or a Linux/macOS environment is the smoothest option.

In [ ]:
# %pip install qiskit qiskit_aer qiskit_ibm_runtime qiskit[visualization] qiskit_nature pyscf qiskit_algorithms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock

from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator

# <b>Defining the Molecule </b> <a id="molecule"></a>
The first step is to describe the physical system. A molecule is specified by the positions of its nuclei, its total charge, and its spin. For H$_2$ we place two hydrogen atoms along the $z$-axis, separated by their **equilibrium bond distance** of about $0.735$ Å, the separation at which the molecule is most stable.

We also choose a **basis set**, which defines the set of atomic orbitals used to represent the electrons. Here we use the minimal `sto3g` basis: with two hydrogen atoms it gives two spatial orbitals, i.e. four spin-orbitals, and therefore a small problem that is perfect for a first VQE.

The `PySCFDriver` runs a classical Hartree-Fock calculation behind the scenes and hands back an `ElectronicStructureProblem`. This object contains the **second-quantized Hamiltonian**

$$
\hat{H} = \sum_{pq} h_{pq}\, a_p^\dagger a_q + \tfrac{1}{2}\sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s,
$$

where the coefficients $h_{pq}$ and $h_{pqrs}$ are the one- and two-electron integrals computed from the chosen geometry and basis, and $a_p^\dagger$, $a_q$ are fermionic creation and annihilation operators.

In [ ]:
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

problem = driver.run()

print("Number of spatial orbitals:", problem.num_spatial_orbitals)
print("Number of particles (alpha, beta):", problem.num_particles)
print("Nuclear repulsion energy:", problem.nuclear_repulsion_energy)

# <b>Mapping to Qubits: Jordan-Wigner </b> <a id="mapping"></a>
The Hamiltonian above is written in terms of **fermionic** operators, but a quantum computer works with **qubits**. We therefore need a *fermion-to-qubit mapping* that translates fermionic creation/annihilation operators into Pauli operators ($X$, $Y$, $Z$, $I$) acting on qubits, while preserving the anticommutation relations that make electrons behave like electrons.

The **Jordan-Wigner** transformation is the most direct such mapping: it assigns **one qubit per spin-orbital**, so the state $|1\rangle$ means "this spin-orbital is occupied" and $|0\rangle$ means "empty". For our H$_2$ problem in the `sto3g` basis, four spin-orbitals map onto **four qubits**.

Applying the mapper turns the fermionic Hamiltonian into a weighted sum of Pauli strings,

$$
\hat{H} = \sum_j c_j \, \hat{P}_j, \qquad \hat{P}_j \in \{I, X, Y, Z\}^{\otimes n},
$$

which is exactly the form whose expectation value a quantum computer can estimate.

In [ ]:
mapper = JordanWignerMapper()

# The fermionic Hamiltonian ("second-quantized operator") for this problem:
fermionic_op = problem.hamiltonian.second_q_op()

# Map it to a qubit (Pauli) operator:
qubit_op = mapper.map(fermionic_op)

print("Number of qubits:", qubit_op.num_qubits)
print("Number of Pauli terms:", len(qubit_op))
print(qubit_op)

# <b>The Ansatz: Hartree-Fock + UCCSD </b> <a id="ansatz"></a>
VQE needs a **parameterized trial state**, the ansatz. A good ansatz should be expressive enough to reach the true ground state, yet not so large that the optimizer gets lost. For chemistry, a physically motivated and widely used choice is the **Unitary Coupled Cluster with Singles and Doubles (UCCSD)** ansatz.

The idea is to start from a good, cheap approximation and improve it:

- **Hartree-Fock (HF) reference state.** This is the mean-field solution, where each electron occupies the lowest available spin-orbitals. On qubits it is simply the computational basis state with the occupied orbitals set to $|1\rangle$. It is our starting point.
- **UCCSD excitations.** On top of HF, UCCSD applies parameterized *single* and *double* excitations, which move electrons from occupied into virtual (empty) orbitals. Formally the trial state is

$$
|\psi(\vec{\theta})\rangle = e^{\hat{T}(\vec{\theta}) - \hat{T}^\dagger(\vec{\theta})} \, |\text{HF}\rangle,
$$

where $\hat{T}$ collects the single and double excitation operators and $\vec{\theta}$ are the parameters VQE will optimize. Capturing these excitations is what lets the ansatz describe **electron correlation** — the physics that Hartree-Fock alone misses.

In Qiskit Nature, `UCCSD` builds this circuit for us, and we pass it a `HartreeFock` object as its `initial_state`.

In [ ]:
hf_state = HartreeFock(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
)

ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=hf_state,
)

print("Number of qubits:", ansatz.num_qubits)
print("Number of variational parameters:", ansatz.num_parameters)
ansatz.decompose().draw("mpl", fold=-1)

# <b>Running VQE </b> <a id="vqe"></a>
We now have every ingredient VQE needs:

1. An **Estimator primitive**, which evaluates the expectation value $\langle \psi(\vec{\theta}) | \hat{H} | \psi(\vec{\theta})\rangle$. Here we use the `StatevectorEstimator`, an exact, noiseless simulator that is ideal for understanding the algorithm before moving to hardware.
2. The **ansatz**, our UCCSD + Hartree-Fock trial state.
3. A **classical optimizer**, which proposes new parameters to lower the energy. We use `SLSQP`, a gradient-based optimizer that works well for smooth, low-dimensional problems like this one.

A natural starting point for the parameters is all zeros: this corresponds to applying no excitations, so the trial state is exactly the Hartree-Fock state. The optimizer then explores from there. Because our system is tiny, we can also compute the **exact** ground-state energy with a classical eigensolver (`NumPyMinimumEigensolver`) and use it to check the VQE result.

The `GroundStateEigensolver` from Qiskit Nature conveniently ties the mapper and the solver together and, importantly, **adds back the nuclear repulsion energy** so that the number we get out is the total molecular energy.

In [ ]:
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

# --- Exact classical reference ---
numpy_solver = NumPyMinimumEigensolver()
exact_calc = GroundStateEigensolver(mapper, numpy_solver)
exact_result = exact_calc.solve(problem)
exact_energy = exact_result.total_energies[0]

# --- VQE ---
estimator = StatevectorEstimator()
optimizer = SLSQP()

vqe = VQE(estimator, ansatz, optimizer)
vqe.initial_point = np.zeros(ansatz.num_parameters)

vqe_calc = GroundStateEigensolver(mapper, vqe)
vqe_result = vqe_calc.solve(problem)
vqe_energy = vqe_result.total_energies[0]

print(f"Exact ground-state energy : {exact_energy:.6f} Ha")
print(f"VQE  ground-state energy  : {vqe_energy:.6f} Ha")
print(f"Absolute error            : {abs(vqe_energy - exact_energy):.2e} Ha")

# <b>Results and Evaluation </b> <a id="res"></a>
If everything went well, the VQE energy should agree with the exact diagonalization to well within **chemical accuracy**, usually defined as $1.6 \times 10^{-3}$ Hartree ($\approx 1$ kcal/mol). This is the precision threshold chemists care about, because it is roughly the accuracy needed to predict reaction energies reliably.

The comparison below shows the VQE energy against the exact value. On a noiseless simulator with the UCCSD ansatz, the two should sit essentially on top of each other, confirming that our ansatz is expressive enough to capture the ground state of H$_2$. The Hartree-Fock energy is included as a baseline so you can *see* how much energy the correlation captured by UCCSD recovers.

In [ ]:
# Hartree-Fock total energy: evaluate the mapped Hamiltonian on the HF state.
# (This is the mean-field baseline, before adding UCCSD correlation.)
from qiskit.primitives import StatevectorEstimator

hf_circuit = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
_est = StatevectorEstimator()
_elec_hf = _est.run([(hf_circuit, qubit_op)]).result()[0].data.evs
hf_energy = float(_elec_hf) + problem.nuclear_repulsion_energy

labels = ["Hartree-Fock\n(baseline)", "VQE\n(UCCSD)", "Exact\n(diagonalization)"]
energies = [hf_energy, vqe_energy, exact_energy]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, energies, color=["#B0B0B0", "#1f77b4", "#2ca02c"])
plt.ylabel("Total energy (Hartree)")
plt.title("Ground-state energy of H$_2$ at equilibrium (0.735 \u00c5)")
plt.axhline(exact_energy, color="#2ca02c", ls="--", lw=1, alpha=0.7)

for bar, e in zip(bars, energies):
    plt.text(bar.get_x() + bar.get_width()/2, e, f"{e:.4f}",
             ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

print(f"Correlation energy recovered by VQE: {hf_energy - vqe_energy:.6f} Ha")

# <b>Moving forward </b> <a id="move"></a>
The workflow in this notebook — describe the molecule, map to qubits, choose an ansatz, run VQE — is intentionally minimal. We computed a **single point**: the ground-state energy of H$_2$ at one fixed bond distance, on a noiseless simulator, with a textbook ansatz and mapping. Real chemistry problems ask for more, and that is where the hackathon challenge begins.

The three levels below are deliberately open-ended. They tell you *what* to aim for, not *how* to get there: choosing the geometry, ansatz, mapping, optimizer, and hardware strategy is part of the challenge. Each level builds naturally on the one before.

### <b>Beginner — a potential energy surface</b>
A single energy tells us little; the interesting physics lives in how the energy *changes* with geometry. Your goal is to compute the **potential energy surface (PES)** of the **HeH$^+$ molecular ion** — its total ground-state energy as a function of the bond distance between He and H. Sweep the internuclear distance across a range, run the ground-state calculation at each point, and plot the resulting curve. Can you locate the equilibrium bond length (the minimum of the curve) and estimate the dissociation behaviour as the atoms are pulled apart? Think about what changes as you move away from equilibrium, and whether your chosen ansatz stays accurate everywhere along the curve.

### <b>Intermediate — do more with less</b>
UCCSD on a full spin-orbital mapping is accurate but expensive, and the cost grows quickly with system size. Move to a **larger molecule** (for example LiH or BeH$_2$) and confront the problem of **reducing the quantum resources** needed to reach chemical accuracy: fewer qubits, shorter circuits, or fewer measurements. There are several levers to explore — alternative fermion-to-qubit mappings, exploiting molecular symmetries to taper qubits away, freezing core orbitals, or grouping commuting measurements — and part of the challenge is deciding which combination buys you the most under a fixed budget of shots or circuit depth.

### <b>Advanced — beyond a single ansatz</b>
The expressiveness of a fixed ansatz eventually becomes a bottleneck: for larger or more strongly correlated systems, a single variational circuit may not reach the ground state, or may become untrainable. Explore approaches that **let classical post-processing shoulder more of the load** — for instance, sampling electronic configurations from quantum circuits and diagonalizing within the sampled subspace, or other hybrid quantum-classical strategies that trade a hard-to-optimize deep circuit for many shallower ones. Push toward a system where a naive UCCSD run struggles, and show that your approach does better. What is the largest or most correlated system you can treat while keeping the results trustworthy?

### <b>Running on real hardware</b>
Whichever level you tackle, moving from the `StatevectorEstimator` to a real IBM Quantum device (such as `ibm_basquecountry`) changes the game. The heavy-hex connectivity, finite shot counts, and gate noise all affect the energy you measure. You are encouraged to compare ideal simulation, noisy simulation, and real-device execution, and to explore **error suppression and mitigation** techniques — measurement-error mitigation, zero-noise extrapolation, dynamical decoupling — as well as **hardware-aware transpilation** to keep circuits shallow. A good solution balances chemical accuracy against what the hardware can actually deliver.

# <b>Useful resources </b> <a id="use"></a>

The following resources may be useful when extending this introductory VQE example toward the full challenge.

### <b>Qiskit Nature (quantum chemistry) </b>

- [Qiskit Nature documentation](https://qiskit-community.github.io/qiskit-nature/)  
  Entry point for building electronic-structure problems, drivers, mappers, and chemistry ansätze.

- [Ground-state solvers tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/03_ground_state_solvers.html)  
  Step-by-step tutorial on finding molecular ground states with VQE and classical solvers — the closest reference to this notebook.

- [Electronic structure problem tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/01_electronic_structure.html)  
  Explains drivers, basis sets, and how the second-quantized Hamiltonian is constructed.

- [Qubit mappers tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/06_qubit_mappers.html)  
  Compares Jordan-Wigner, parity, and Bravyi-Kitaev mappings, including qubit tapering via symmetries — directly relevant to the intermediate challenge.

- [UCCSD ansatz documentation](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.circuit.library.UCCSD.html)  
  Reference page for the UCCSD variational form used in this notebook.

### <b>Variational algorithms and optimizers </b>

- [VQE documentation](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.VQE.html)  
  Reference page for the Variational Quantum Eigensolver class.

- [Qiskit Algorithms optimizers](https://qiskit-community.github.io/qiskit-algorithms/apidocs/qiskit_algorithms.optimizers.html)  
  Reference page for classical optimizers such as `SLSQP`, `COBYLA`, `SPSA`, and others to experiment with.

- [AdaptVQE how-to](https://qiskit-community.github.io/qiskit-nature/howtos/adapt_vqe.html)  
  Builds an ansatz adaptively, a useful stepping stone toward the advanced challenge.

- [Variational quantum algorithms course](https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/variational-quantum-algorithms)  
  IBM Quantum learning material on the broader family of hybrid variational algorithms.

### <b>Hardware execution, transpilation, and error mitigation </b>

- [Qiskit documentation](https://quantum.cloud.ibm.com/docs/guides/tools-intro)  
  General entry point for circuits, primitives, transpilation, and runtime execution on IBM Quantum.

- [Transpile with pass managers](https://quantum.cloud.ibm.com/docs/guides/transpile-with-pass-managers)  
  How to transpile circuits for a specific backend and its connectivity.

- [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques)  
  Overview of the resilience options available through Qiskit Runtime.

- [Combine error mitigation options with the Estimator primitive](https://quantum.cloud.ibm.com/docs/tutorials/combine-error-mitigation-techniques)  
  Tutorial on dynamical decoupling, measurement-error mitigation, gate twirling, and zero-noise extrapolation.

### <b>Background reading </b>

- [Kandala et al., *Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets*, Nature (2017)](https://www.nature.com/articles/nature23879)  
  The IBM Quantum paper that inspires this challenge track.

## <b>Credits and license</b>

This notebook was written by **Benjamin Tirado** for the **BasQ Qiskit Fall Fest 2026**, organised by Basque Quantum (BasQ).

© 2026 Benjamin Tirado. The text, figures and explanations are released under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); the code is released under the
[Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0), the same license as Qiskit.

You are welcome to run, adapt and share this material — including as a starting point for your own
challenge submission — provided the attribution above is kept. If you reuse it publicly, please cite it as:

> B. Tirado, *Ground-State Energy of a Molecule with VQE*, tutorial notebook, BasQ Qiskit Fall Fest, 2026.

Built with [Qiskit](https://www.ibm.com/quantum/qiskit), [Qiskit Nature](https://qiskit-community.github.io/qiskit-nature/)
and [PySCF](https://pyscf.org/), which remain the property of their respective authors and are used
under their own licenses.

*Questions, corrections or suggestions:* benjamin.tirado@ehu.eus